# Python Variables and Mutability: How Memory Really Works

Learn how Python stores variables in memory, the exact difference between mutable and immutable data types, and why list assignment b = a doesn't create a copy. We cover memory pointers, reference counts, shallow versus deep copies, and mutable default function arguments.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aquashubham/neuralnest-code/blob/main/python-for-ai/variables-data-types-and-mutability-how-python-really-stores.ipynb)

Part of the **Python for AI** series on NeuralNest.

Run the code cells from top to bottom. Change the numbers and run them again: that's the fastest way to really understand it.

## Why this matters

When I first started writing Python, I ran into a bug that cost me an entire hour. I assigned a list to a new variable, updated the new list, and was shocked to see my original list change too. If you are coming from languages like C or Java, or if you are brand new to programming, Python variables do not work the way you think. In this video, part of our Python for AI series, you will learn how Python actually stores objects in memory, what mutability really means, and how to avoid the hidden traps that catch developers when building data pipelines and machine learning models. By the end, you will predict exactly how Python handles memory assignment every single time.

## Variables as Name Tags, Not Boxes

1. Memory Space
2. Object 42
3. Tag x
4. Tag y

Many tutorials tell you that a variable is a box that holds a value. In Python, that model is completely wrong. Think of memory as a vast storage room filled with objects. A Python variable is not a box containing a number or text. It is simply a sticky label or name tag attached to an object in memory. When you create an object, Python allocates memory for it and creates a specific memory address. When you assign that object to a variable name, Python just sticks the label onto that object. Multiple labels can be attached to the exact same object without making any extra copy of the data.

## Inspecting Object Identity

Let us inspect how Python tracks object identity in code. We assign the integer 42 to x, and then set y equal to x. In Python, the built in id function returns the unique integer representing an object memory address. When we print the identity of x and y, notice that both numbers are identical. The expression x is y checks whether both variables point to the exact same object in memory, returning True. Notice we did not create two numbers. We created one integer object in memory, and stuck two labels, x and y, onto it.

In [ ]:
x = 42
y = x
print(id(x))
# -> 140722394851216
print(id(y))
# -> 140722394851216
print(x is y)
# -> True

## Object Reference Count in Memory

**Variables and Memory Pointers** (highlighted rows in **bold**)

| Variable Name | Memory Address | Object Value | Ref Count |
|---|---|---|---|
| **x** | **0x7FFF01** | **42** | **2** |
| **y** | **0x7FFF01** | **42** | **2** |
| z | 0x7FFF08 | 99 | 1 |

Here is what Python memory looks like behind the scenes. The highlighted rows show variables x and y sharing memory address 0x7FFF01, which holds integer 42. Python maintains an internal counter called a reference count for every single object. Because both x and y point to integer 42, its reference count is two. Variable z points to a completely separate memory address holding integer 99, with a reference count of one. When an object reference count drops to zero, Python garbage collector automatically frees that memory so you do not have to manage memory by hand.

## Rebinding Immutable Objects

Now what happens when we modify a variable holding a number? We set x to 10, check its identity, and then run x equals x plus 1. The output shows x is now 11, but look closely at the identity. The memory address changed from ending in 192 to 224. Integers in Python are immutable, meaning their internal value can never be altered once created. When you calculate x plus 1, Python builds a brand new integer object 11 at a new memory location and moves the label x to point to it.

In [ ]:
x = 10
print(id(x))
# -> 140722394850192
x = x + 1
print(x)
# -> 11
print(id(x))
# -> 140722394850224

## How Rebinding Changes Memory

1. Original Object 10
2. Compute 10 + 1
3. New Object 11
4. Move Label x

Let us trace this step by step on screen. Step one shows variable x pointing to integer 10. Step two computes 10 plus 1. Because integer objects are immutable, Python cannot edit the number 10 inside its original memory space. In step three, Python allocates a new memory block for integer 11. Finally, step four detaches the label x from 10 and rebinds it to 11. The original integer 10 stays unchanged. If no other variable points to 10, Python garbage collector clears it out.

## Immutable vs Mutable Data Types

- Immutable: int, float, str, tuple, bool, bytes
- Mutable: list, dict, set, bytearray
- Immutable objects cannot be modified in place
- Mutable objects allow in-place value changes

Every data type in Python falls into one of two categories: immutable or mutable. Immutable types include numbers like integers and floats, strings, tuples, booleans, and bytes. Once an immutable object is created, its value inside memory can never be altered. Mutable types include lists, dictionaries, sets, and bytearrays. These objects allow you to add, remove, or modify elements inside the same memory address. Understanding this distinction is vital before you write code that processes large datasets or model weights.

## The List Alias Trap

Here is the exact trap that cost me an hour when I started Python. We create a list called scores containing test marks 85, 90, and 95. We assign alias scores equal to scores, hoping to keep a backup copy. Then we append 100 to alias scores. Look at the print output for our original scores list. It now contains 100 as well. The identity test returns True because alias scores is not a copy. It is just a second label attached to the exact same list in memory. Modifying the object through either label changes the underlying data for both.

In [ ]:
scores = [85, 90, 95]
alias_scores = scores
alias_scores.append(100)
print(scores)
# -> [85, 90, 95, 100]
print(scores is alias_scores)
# -> True

## Visualizing Shared Mutable Memory

1. Label scores
2. Label alias_scores
3. Single List Object
4. Updated Elements

Let us see why this happens. We are now halfway through our lesson, and next we will look at how to properly copy data. In this diagram, both labels scores and alias scores point to a single list object in memory. When you invoke the append method, Python updates that specific list in place without creating a new memory object. Because both labels refer to that single object, reading from scores displays the mutated data. What do you think happens if we want an independent copy that does not share memory?

## Shallow vs Deep Copying

To solve this, we can make a copy. But Python gives us two kinds of copies: shallow copy and deep copy. In this code, original contains nested lists. Using original dot copy creates a shallow copy. Using copy dot deepcopy creates a full deep copy. When we append 99 to the first inner list of original, notice what happens. The shallow copy still sees the number 99 because shallow copies duplicate the outer list, but keep references to the inner nested objects. The deep copy duplicates everything recursively, staying completely independent.

In [ ]:
import copy
original = [[1, 2], [3, 4]]
shallow = original.copy()
deep = copy.deepcopy(original)
original[0].append(99)
print(shallow)
# -> [[1, 2, 99], [3, 4]]
print(deep)
# -> [[1, 2], [3, 4]]

## Comparing Assignment vs Shallow vs Deep Copy

**Copy Method Behavior** (highlighted rows in **bold**)

| Operation | Outer Memory | Inner Objects Shared | Modifications Shared |
|---|---|---|---|
| b = a | Same | Yes | All modifications |
| **b = a.copy()** | **New** | **Yes** | **Nested items only** |
| **b = copy.deepcopy(a)** | **New** | **No** | **None (Fully isolated)** |

This table compares all three assignment operations side by side. Simple variable assignment b equals a keeps the same outer memory and shares all modifications. The highlighted rows show the true copying techniques. A shallow copy created with a dot copy allocates a new outer container, but still shares references to inner objects. A deep copy creates new outer and inner objects, ensuring total memory isolation. In machine learning pipelines, failing to use deep copy on nested configuration dictionaries is a frequent source of hidden bugs.

## The Mutable Default Argument Trap

Another major pitfall catches developers in technical interviews: mutable default arguments in functions. Here we define add item with a default parameter target set to an empty list. We call add item with value 1 and get list with 1. We call it again with value 2, expecting a fresh empty list, but get 1 and 2. Python evaluates default arguments once when the function is defined, not each time it is called. That default list object persists across every function call, accumulating values silently.

In [ ]:
def add_item(val, target=[]):
    target.append(val)
    return target

print(add_item(1))
# -> [1]
print(add_item(2))
# -> [1, 2]
print(add_item(3))
# -> [1, 2, 3]

## Memory Footprint of Immutable vs Mutable

Let us look at why object immutability matters for performance. This bar chart compares memory allocation sizes in bytes for integers, floats, tuples, and lists. Notice that a list containing three elements consumes 88 bytes, while an equivalent tuple with three elements consumes only 64 bytes. Lists require extra memory capacity overhead because they are mutable and must allow dynamic appending. When processing millions of rows in data science workloads, preferring immutable tuples over lists drastically reduces memory consumption and speeds up execution.

In [ ]:
import matplotlib.pyplot as plt

labels = ['int', 'float', 'tuple (3 items)', 'list (3 items)']
width = 0.8 / 1
plt.bar([i - 0.4 + width * (0 + 0.5) for i in range(4)], [28.0, 24.0, 64.0, 88.0], width, label='Size in Bytes')
plt.xticks(range(4), labels)
plt.xlabel('Data Type')
plt.ylabel('Bytes in Memory')
plt.show()

## Key Takeaways

- Variables are name tags bound to objects
- Immutable types move labels on change
- Assignment b = a creates an alias, not copy
- Use deepcopy for nested mutable structures

Let us review what we learned today. First, Python variables are name tags pointing to memory objects, not memory containers. Second, immutable objects like numbers and strings cannot be changed in place; reassigning them moves the label to a new memory address. Third, assigning b equals a creates an alias pointing to the exact same list or dictionary, which can lead to unintentional side effects. Finally, use shallow or deep copy when you need independent data structures. In our next video, we will explore Python function scopes and how variable references behave across function boundaries.